# FedKDL Kaggle Experiments Runner
Notebook này giúp bạn chạy từng bước của quy trình huấn luyện FedKDL một cách dễ dàng, tránh bị ngắt kết nối và dễ dàng theo dõi log.

### 1. Tải Source Code từ GitHub
Clone repo FedKDL mới nhất từ GitHub và di chuyển vào thư mục làm việc.

In [ ]:
!git clone https://github.com/ngnam1104/FedKDL.git
import os
os.chdir('/kaggle/working/FedKDL')
print('Current Working Directory:', os.getcwd())

### 2. Cài đặt thư viện Cài đặt thư viện và Khởi tạo đường dẫn


In [ ]:
!pip install ultralytics pandas pyyaml torch torchvision
import os, sys
sys.path.append(os.getcwd())

!mkdir -p /kaggle/working/FedKDL/datasets/URPC2020
!ln -s /kaggle/input/datasets/lywang777/urpc2020/URPC2020 /kaggle/working/FedKDL/datasets/URPC2020/URPC2020



### 3. Tạo Topologies và Data Partitions

In [ ]:
!python utils/generate_all_envs.py --n 30 --dataset URPC --m-relays 5 --alphas 1.0

### 4.1. Pre-train Centralized Full-Finetune (Upper Bound)
Chạy Centralized Full Finetune trên toàn bộ dữ liệu để có baseline Upper Bound cho cấu trúc gốc không dùng LoRA.

In [ ]:
!python scripts/fedkdl/train_student_warmup.py --mode centralized_full
# Nếu bị ngắt giữa chừng, đổi thành lệnh dưới để chạy tiếp:
# !python scripts/fedkdl/train_student_warmup.py --mode centralized_full --resume

### 4.2. Pre-train Centralized LoRA
Chạy Centralized LoRA trên toàn bộ dữ liệu để so sánh hiệu năng trực tiếp với Full Finetune.

In [ ]:
!python scripts/fedkdl/train_student_warmup.py --mode centralized_lora
# Nếu bị ngắt giữa chừng, đổi thành lệnh dưới để chạy tiếp:
# !python scripts/fedkdl/train_student_warmup.py --mode centralized_lora --resume

### 5. Khởi chạy thuật toán FedKDL (Main Baseline)
**Giải đáp về Log:** Trong lệnh dưới, toán tử `2>&1 | tee ...` sẽ vừa in log ra giao diện Kaggle Notebook để bạn xem trực tiếp, VỪA ghi song song toàn bộ log vào file `.log`. Kể cả khi Notebook của bạn bị reload/treo trình duyệt, file log vật lý trên hệ thống Kaggle vẫn được lưu bình thường không sót chữ nào!

In [ ]:
!pip uninstall -y ray

In [ ]:
!mkdir -p results/logs_kdl results/train_logs/kdl

!python main_trainer_od.py \
    --topo environments/2d/topo/N_30/topo_N30_seed1104.pkl \
    --data environments/2d/data/URPC/N_30/data_N30_URPC_a1p0_seed1104.pkl \
    --baseline fedkdl \
    --rounds 60 \
    --out-dir results/logs_kdl \
    --log-dir results/train_logs/kdl \
    2>&1 | tee results/train_logs/kdl/kaggle_fedkdl_raw.log

### 6. Khởi chạy các Kịch bản So sánh (Ablation)
Bạn có thể đổi `baseline` thành `fedkdl_selective`, `fedprox_kdl`, `fedkd` tùy nhu cầu để chạy các thử nghiệm khác.

In [ ]:
!python main_trainer_od.py \
    --topo environments/2d/topo/N_30/topo_N30_seed1104.pkl \
    --data environments/2d/data/URPC/N_30/data_N30_URPC_a1p0_seed1104.pkl \
    --baseline fedkd \
    --rounds 60 \
    --out-dir results/logs_kdl \
    --log-dir results/train_logs/kdl \
    2>&1 | tee results/train_logs/kdl/kaggle_fedkd_raw.log